# Semiconductor Wafer Yield Prediction

**Dataset:** UCI SECOM Dataset  
**Goal:** Predict wafer Pass/Fail early in manufacturing to save costs  
**Business Value:** Early defect detection saves $1,500/wafer in manufacturing costs

## Workflow:
1. Data Loading & Exploration
2. Missing Data Analysis & Handling
3. Variance-Based Feature Selection
4. Dimensionality Reduction (PCA)
5. Class Imbalance Handling (SMOTE)
6. Model Training (XGBoost)
7. Threshold Tuning & Evaluation

## 1. Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Imbalanced learning
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Model
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

## 2. Load and Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('uci-secom.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()[:10]}...") 
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check class distribution
print("Class Distribution:")
print(df['Pass/Fail'].value_counts())
print(f"\nImbalance Ratio: {df['Pass/Fail'].value_counts()[-1] / df['Pass/Fail'].value_counts()[1]:.1f}:1")

## 3. Missing Data Analysis

In [ ]:
# Analyze missing data
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Percentage': (df.isnull().sum() / len(df) * 100).values
}).sort_values('Percentage', ascending=False)

print(f"Columns with >40% missing data: {(missing_data['Percentage'] > 40).sum()}")
print(f"\nTop 10 columns with most missing data:")
missing_data.head(10)

In [ ]:
# Visualize missing data distribution
plt.figure(figsize=(10, 6))
plt.hist(missing_data['Percentage'], bins=range(0, 110, 10), edgecolor='black')
plt.xlabel('Missing Data Percentage')
plt.ylabel('Number of Columns')
plt.title('Distribution of Missing Data Across Sensors')
plt.grid(axis='y', alpha=0.3)
plt.show()

## 4. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop(['Time', 'Pass/Fail'], axis=1)
y = df['Pass/Fail']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target classes: {y.unique()}")

In [ ]:
# Step 1: Remove columns with >40% missing data
cols_to_drop = missing_data[missing_data['Percentage'] > 40]['Column'].tolist()
cols_to_drop = [col for col in cols_to_drop if col in X.columns]  # Only sensor columns

X_filtered = X.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns with >40% missing data")
print(f"Remaining shape: {X_filtered.shape}")

In [ ]:
# Step 2: Impute remaining missing values with median
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_filtered),
    columns=X_filtered.columns,
    index=X_filtered.index
)

print(f"Remaining NaN values: {X_imputed.isnull().sum().sum()}")
print(f"Final shape after imputation: {X_imputed.shape}")

## 5. Variance-Based Feature Selection

In [ ]:
# Calculate variance for all features
variance = X_imputed.var()

# Visualize variance distribution (log scale)
plt.figure(figsize=(10, 6))
variance_sorted = variance.sort_values()
plt.plot(range(len(variance_sorted)), np.log10(variance_sorted))
plt.xlabel('Sensor Index (sorted by variance)')
plt.ylabel('Log10(Variance)')
plt.title('Variance Distribution Across Sensors (Log Scale)')
plt.grid(True, alpha=0.3)
plt.show()

# Print statistics
print("\nVariance Statistics:")
print(f"Min:    {variance.min():.2e}")
print(f"Q1:     {variance.quantile(0.25):.2e}")
print(f"Median: {variance.median():.2e}")
print(f"Q3:     {variance.quantile(0.75):.2e}")
print(f"Max:    {variance.max():.2e}")

In [ ]:
# Remove low and high variance features
threshold_low = 0.001   # Remove near-constant features
threshold_high = 10000  # Remove extremely noisy features

cols_to_keep = variance[(variance >= threshold_low) & (variance <= threshold_high)].index
X_variance_filtered = X_imputed[cols_to_keep]

print(f"Filtering criteria:")
print(f"  Remove variance < {threshold_low} (too constant)")
print(f"  Remove variance > {threshold_high:.0e} (too noisy)")
print(f"\nResult: {X_imputed.shape[1]} → {X_variance_filtered.shape[1]} columns")
print(f"Removed: {X_imputed.shape[1] - X_variance_filtered.shape[1]} columns")

## 6. Train-Test Split

In [ ]:
# Split data (80/20 split, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_variance_filtered, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())

## 7. Build ML Pipeline with PCA, SMOTE, and XGBoost

In [ ]:
# Convert labels for XGBoost (expects 0/1, we have -1/1)
y_train_xgb = np.where(y_train == -1, 0, 1)
y_test_xgb = np.where(y_test == -1, 0, 1)

# Build pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),              # Standardize features
    ('pca', PCA(n_components=0.95)),           # Keep 95% variance
    ('smote', SMOTE(random_state=42)),         # Balance classes
    ('classifier', XGBClassifier(
        scale_pos_weight=100,                   # Heavy weight on minority class
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss'
    ))
])

print("Pipeline created successfully!")
print("\nPipeline steps:")
for name, step in pipeline.steps:
    print(f"  {name}: {type(step).__name__}")

In [ ]:
# Train the pipeline
print("Training model...")
pipeline.fit(X_train, y_train_xgb)
print("Training complete!")

# Check PCA components
n_components = pipeline.named_steps['pca'].n_components_
explained_var = pipeline.named_steps['pca'].explained_variance_ratio_.sum()
print(f"\nPCA reduced {X_train.shape[1]} features → {n_components} components")
print(f"Total variance explained: {explained_var:.2%}")

## 8. Model Evaluation

In [ ]:
# Make predictions
y_pred_xgb = pipeline.predict(X_test)
y_pred = np.where(y_pred_xgb == 0, -1, 1)  # Convert back to original labels

# Classification report
print("Classification Report:")
print("="*70)
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print(f"\nBreakdown (Pass=-1 as POSITIVE class):")
print(f"  True Positives (Pass correct):  {cm[0,0]}")
print(f"  False Negatives (Pass→Fail):    {cm[0,1]} (false alarms)")
print(f"  False Positives (Fail→Pass):    {cm[1,0]} (MISSED DEFECTS!)")
print(f"  True Negatives (Fail correct):  {cm[1,1]}")

## 9. Threshold Tuning

In [ ]:
# Get probability predictions
y_proba = pipeline.predict_proba(X_test)

# Test different thresholds
print("Testing different decision thresholds:")
print("="*70)

for threshold in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    # Predict based on threshold
    y_pred_thresh_xgb = (y_proba[:, 1] > threshold).astype(int)
    y_pred_thresh = np.where(y_pred_thresh_xgb == 0, -1, 1)
    
    # Get metrics
    report = classification_report(y_test, y_pred_thresh, zero_division=0, output_dict=True)
    
    # Calculate key metrics (Fail = class 1 = negative class in your view)
    fail_recall = report['1']['recall']  # Specificity
    fail_precision = report['1']['precision']
    defects_caught = int(fail_recall * 21)  # Assuming 21 defects in test set
    
    # Count false alarms
    total_fail_preds = (y_pred_thresh == 1).sum()
    false_alarms = total_fail_preds - defects_caught
    
    print(f"\n📊 Threshold: {threshold}")
    print(f"   Defects caught: {defects_caught}/21 (Recall: {fail_recall:.2f})")
    print(f"   False alarms:   {false_alarms} good wafers flagged")
    print(f"   Precision:      {fail_precision:.2f}")
    print(f"   Cost-benefit:   Save ${defects_caught*1500} - Waste ${false_alarms*100} = ${defects_caught*1500 - false_alarms*100}")

## 10. Business Impact Summary

In [ ]:
# Calculate business impact at different thresholds
print("Business Impact Analysis")
print("="*70)
print("\nAssumptions:")
print("  - Early stop saves: $1,500 per defect")
print("  - False alarm cost: $100 per good wafer stopped")
print("  - Missed defect cost: $1,600 per defect (full manufacturing)")
print("\nRecommended threshold: 0.10-0.20 for best cost-benefit trade-off")
print("\nKey Insight: Catching defects early is more valuable than avoiding false alarms!")